# Spark Memory Management & Garbage Collection

## JVM Memory Regions — The Full Picture

Every Spark executor is a JVM process. Understanding its memory layout is the prerequisite for diagnosing OOM errors, spill, and GC pauses.

```
┌─────────────────────────────────────────────────────────────────┐
│                     EXECUTOR JVM PROCESS                        │
│                                                                 │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │                   ON-HEAP MEMORY                         │   │
│  │   (controlled by -Xmx, i.e. spark.executor.memory)      │   │
│  │                                                          │   │
│  │  ┌───────────────────────────────────────────────────┐   │   │
│  │  │  Unified Memory Pool  (spark.memory.fraction=0.6) │   │   │
│  │  │                                                   │   │   │
│  │  │   ┌─────────────────────┐ ┌────────────────────┐  │   │   │
│  │  │   │  Execution Memory   │ │  Storage Memory    │  │   │   │
│  │  │   │  (shuffles, sorts,  │ │  (cached RDDs,     │  │   │   │
│  │  │   │   aggregations,     │ │   broadcast vars,  │  │   │   │
│  │  │   │   joins)            │ │   accumulators)    │  │   │   │
│  │  │   │  storageFraction    │ │  storageFraction   │  │   │   │
│  │  │   │  boundary is soft   │ │  boundary is soft  │  │   │   │
│  │  │   └─────────────────────┘ └────────────────────┘  │   │   │
│  │  └───────────────────────────────────────────────────┘   │   │
│  │                                                          │   │
│  │  ┌───────────────────────────────────────────────────┐   │   │
│  │  │  User Memory  (1 - spark.memory.fraction = 0.4)  │   │   │
│  │  │  Spark internal metadata, user data structures,  │   │   │
│  │  │  UDF closures, HashMaps in application code      │   │   │
│  │  └───────────────────────────────────────────────────┘   │   │
│  │                                                          │   │
│  │  ┌───────────────────────────────────────────────────┐   │   │
│  │  │  Reserved Memory  (fixed ~300 MB, Spark internals)│   │   │
│  │  └───────────────────────────────────────────────────┘   │   │
│  └──────────────────────────────────────────────────────────┘   │
│                                                                 │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │               OFF-HEAP MEMORY (optional)                 │   │
│  │   spark.memory.offHeap.enabled = true                    │   │
│  │   spark.memory.offHeap.size = Ng                         │   │
│  │   Allocated via sun.misc.Unsafe; NOT subject to GC       │   │
│  │   Used for: Tungsten columnar format, certain caches     │   │
│  └──────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

**Why does this architecture exist?**  
Before Spark 1.6, execution and storage memory were *statically partitioned*. If your job needed lots of shuffle memory but cached little, the storage pool sat idle while execution spilled to disk. The **Unified Memory Model** (introduced in 1.6) made the boundary *soft*: either region can borrow from the other as long as the owning region is not under pressure. This dramatically reduced the need for manual memory tuning.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("MemoryManagement_GC")
    # Executor heap size — the -Xmx value passed to the JVM
    .config("spark.executor.memory", "2g")
    # spark.memory.fraction: fraction of (heap - 300MB reserved) used by Spark
    # The remaining (1 - fraction) = 0.4 is User Memory
    .config("spark.memory.fraction", "0.6")
    # spark.memory.storageFraction: within the Spark pool, fraction guaranteed
    # to storage (cached data); execution can still use it when storage is idle
    .config("spark.memory.storageFraction", "0.5")
    # AQE helps avoid spill by coalescing partitions
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "100")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} started")
print(f"executor.memory     : {spark.conf.get('spark.executor.memory')}")
print(f"memory.fraction     : {spark.conf.get('spark.memory.fraction')}")
print(f"memory.storageFraction: {spark.conf.get('spark.memory.storageFraction')}")

## Executor Memory Layout — The Numbers

Let's make the fractions concrete with a worked example using the defaults (2 GB executor, fraction=0.6, storageFraction=0.5).

```
spark.executor.memory = 2048 MB

Step 1 — subtract Reserved Memory:
  usable = 2048 MB - 300 MB = 1748 MB

Step 2 — split by spark.memory.fraction = 0.6:
  Spark pool = 1748 × 0.6  =  1048 MB
  User pool  = 1748 × 0.4  =   699 MB

Step 3 — split Spark pool by spark.memory.storageFraction = 0.5:
  Storage floor  = 1048 × 0.5  =  524 MB  (can be invaded by Execution)
  Execution pool = 1048 × 0.5  =  524 MB  (can borrow from Storage)
  (both pools can use the other's space if idle)
```

```
 2048 MB executor heap
 ├── 300 MB   Reserved (Spark internals)
 ├── 699 MB   User Memory  (your UDFs, HashMaps, application objects)
 └── 1048 MB  Unified Pool
      ├── ~524 MB  Storage  (RDD cache, broadcast variables)
      └── ~524 MB  Execution (shuffle buffers, sort runs, aggregation maps)
                   ↕ (soft boundary — either side can borrow)
```

### Key Interactions

- **Execution evicts Storage**: if execution needs more space and storage holds cached blocks, those blocks are evicted (dropped from RAM, recomputed on next access).
- **Storage cannot evict Execution**: ongoing shuffle/sort operations cannot be interrupted. This asymmetry is intentional — a running task must complete; a cache hit is a nice-to-have.
- **`spark.memory.storageFraction` is a floor, not a ceiling**: storage is *guaranteed* that fraction but can use more if execution is idle.

In [ ]:
# ── Inspecting Memory via statusTracker and REST API ─────────────────────────
#
# The SparkContext.statusTracker() provides programmatic access to executor info.
# For detailed memory metrics (heap used, GC time, spill bytes) you need the
# Spark REST API or the Spark UI.
#
# REST API endpoint pattern:
#   http://<driver-host>:4040/api/v1/applications/<app-id>/executors
#
# Example response fields per executor:
#   memoryUsed          → bytes of storage memory in use
#   maxMemory           → total storage pool available
#   totalGCTime         → cumulative JVM GC milliseconds
#   totalShuffleRead    → bytes read from shuffle
#   totalShuffleWrite   → bytes written to shuffle
#   memoryMetrics.usedOnHeapStorageMemory
#   memoryMetrics.usedOffHeapStorageMemory
#   memoryMetrics.totalOnHeapStorageMemory
#   memoryMetrics.totalOffHeapStorageMemory

sc = spark.sparkContext
tracker = sc.statusTracker()

# List active executor IDs (driver is excluded from the worker list)
executor_ids = tracker.getExecutorInfos()
print("Active executors reported by statusTracker():")
for ei in executor_ids:
    print(f"  Host: {ei.host()}  Cores: {ei.numRunningTasks()}")

# Show how to construct the REST URL programmatically
app_id = sc.applicationId
ui_port = sc.uiWebUrl  # e.g. http://driver-host:4040
rest_url = f"{ui_port}/api/v1/applications/{app_id}/executors"
print()
print(f"Application ID : {app_id}")
print(f"Spark UI URL   : {ui_port}")
print(f"Executors REST : {rest_url}")
print()
print("To query the REST API from within this notebook (requires network access):")
print("  import requests")
print(f"  resp = requests.get('{rest_url}')")
print("  for ex in resp.json():")
print("      print(ex['id'], ex['memoryUsed'], ex['totalGCTime'])")

## Spill — What It Is, How to Detect It, How to Avoid It

### What is Spill?

When a **sort**, **aggregation**, or **shuffle** operation exhausts the execution memory pool, Spark *spills* intermediate data to the executor's local disk. The spilled data is serialised, compressed, and written to a temp file; once the in-memory pass is complete, Spark merges the in-memory and on-disk portions (a merge-sort / external merge pattern).

Two spill metrics appear in the Spark UI:
- **Shuffle Spill (Memory)**: the size of the in-memory data that *was* spilled (pre-serialisation size — useful for estimating how much more memory you'd need to avoid spill).
- **Shuffle Spill (Disk)**: the actual bytes written to disk (post-serialisation, usually smaller due to compression).

### How to Detect Spill

1. **Spark UI → Jobs → Stages → click a stage** — look at the "Shuffle Spill (Memory)" and "Shuffle Spill (Disk)" columns in the task table.
2. **Spark UI → Executors tab** → "Shuffle Spill" column.
3. **Log search**: grep executor logs for `ExternalSorter`, `ExternalAppendOnlyMap`, or `spilling`.

### How to Avoid Spill

| Root cause | Fix |
|---|---|
| Too few partitions → each task processes too much data | Increase `spark.sql.shuffle.partitions`; let AQE coalesce afterward |
| `spark.executor.memory` too low | Increase executor memory; check container limits in YARN/K8s |
| Data skew → one task carries disproportionate data | Enable `spark.sql.adaptive.skewJoin.enabled`; add a salt column to the join key |
| Caching too aggressively → storage evicts execution budget | Unpersist caches before heavy shuffle operations; lower `storageFraction` |
| Large UDF objects in User Memory | Simplify UDFs; use native Spark functions instead |

In [ ]:
# ── Demonstrating Spill Conditions ───────────────────────────────────────────
# In a production cluster you would genuinely trigger spill by constraining
# spark.executor.memory to e.g. 512m and running a large sort.
# Here we simulate the scenario with comments describing what Spark does
# at each step, and deliberately run a sort that WOULD spill in a
# memory-constrained environment.

# For demonstration: use many shuffle partitions but process a relatively
# large dataset so individual tasks are reasonably large.

# In a low-memory scenario (e.g. spark.executor.memory=256m), this sort
# would spill because each partition's sort buffer exceeds available execution memory.
spark.conf.set("spark.sql.shuffle.partitions", "10")  # intentionally few → large tasks

# Create ~40 MB of data (300k rows × ~130 bytes each)
large_df = (
    spark.range(300_000)
    .withColumn("payload",  F.sha2(F.col("id").cast("string"), 256))  # 64-char hex string
    .withColumn("sort_key", F.rand())
)

print("Running sort with few partitions (spill-prone pattern)...")
print("In a real memory-constrained cluster this would generate:")
print("  ExternalSorter: spilling to disk  (in executor logs)")
print("  Shuffle Spill (Memory): N MB  (in Spark UI Stage detail)")
print()

t0 = time.time()
# orderBy triggers a full shuffle sort — the most spill-prone operation
sorted_df = large_df.orderBy("sort_key")
row_count = sorted_df.count()  # materialize
elapsed = time.time() - t0

print(f"Sorted {row_count:,} rows in {elapsed:.2f}s with 10 shuffle partitions")
print()
print("REMEDIATION (what you would do in a real spill scenario):")
print("  1. Increase partitions: spark.sql.shuffle.partitions = 200")
print("     → Each task processes less data → less pressure on sort buffer")
print("  2. Increase executor memory: spark.executor.memory = 4g")
print("  3. Use Kryo serialiser to reduce object size in memory")
print("  4. Avoid global sort (orderBy) — sort within partitions (sortWithinPartitions)")

# Reset
spark.conf.set("spark.sql.shuffle.partitions", "100")

## GC Pressure — Symptoms, Causes, and Fixes

### Symptoms

The clearest signal is in **Spark UI → Executors tab → "GC Time"** column. If an executor spends more than ~10% of its total task time in garbage collection, GC is a bottleneck. In extreme cases you will also see:
- `java.lang.OutOfMemoryError: Java heap space` in executor logs.
- `java.lang.OutOfMemoryError: GC overhead limit exceeded` — JVM giving up because it spent > 98% of CPU time collecting garbage.
- Tasks running much slower than they should based on data volume.

### Root Causes

1. **Many short-lived objects** — Row-by-row UDFs in Python (via Py4J), iterating over Row objects in Scala/Java, or complex lambda closures create enormous numbers of small Java objects. The young generation fills up, triggering frequent minor GC.

2. **Large long-lived objects** — DataFrames cached in storage memory as Java objects (the deserialized `MEMORY_ONLY` storage level) hold entire object graphs in the old generation. Full GC (stop-the-world) pauses can last seconds.

3. **Java serializer for shuffle data** — The default `java.io.ObjectOutputStream` serializer creates deeply nested object graphs and is slow. Shuffle data sits in memory during the write phase, adding heap pressure.

### Fixes

| Fix | How it helps |
|---|---|
| **Kryo serializer** | 2–5x faster serialisation; ~20–30% less memory for shuffle data; fewer intermediate objects |
| **Off-heap storage** | `spark.memory.offHeap.enabled=true` — cached data lives outside the JVM heap, invisible to GC |
| **MEMORY_AND_DISK_SER cache level** | Serialised cache objects are single byte arrays, not object trees — much lighter on GC |
| **Reduce caching** | Unpersist DataFrames you no longer need; GC cannot collect what you still reference |
| **Native Spark functions over UDFs** | `F.upper()` runs in JVM codegen with minimal object creation; a Python UDF round-trips through Py4J |
| **G1GC tuning** | `-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35` — G1 handles large mixed workloads better than the default collector |

In [ ]:
# ── Kryo Serializer vs Java Serializer ───────────────────────────────────────
# Kryo is a fast binary serialisation library.  When used as Spark's shuffle
# serializer it:
#   - Produces smaller byte arrays (less shuffle I/O)
#   - Creates fewer intermediate objects (less GC pressure)
#   - Is 2–5x faster to encode/decode than Java serialisation
#
# IMPORTANT: Kryo only benefits SHUFFLE data (task-to-task transfer and spill).
# DataFrame/Dataset operations already use Tungsten binary format internally,
# so Kryo's biggest wins are in:
#   - rdd.map / rdd.flatMap (old RDD API)
#   - persist(StorageLevel.MEMORY_ONLY_SER)
#   - Custom objects passed via broadcast variables

# NOTE: In Spark 3.x, changing spark.serializer after the session is created
# has no effect on the running session.  This code shows what you would set
# in SparkSession.builder before .getOrCreate().

print("=" * 60)
print("CURRENT serializer:")
print(f"  {spark.conf.get('spark.serializer', 'org.apache.spark.serializer.JavaSerializer')}")
print()
print("TO USE KRYO — set these before calling .getOrCreate():")
print()
print("  SparkSession.builder")
print("    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')")
print("    # Kryo registration avoids class-name overhead per object")
print("    .config('spark.kryo.registrationRequired', 'false')  # lenient mode")
print("    # Register your custom classes for maximum efficiency:")
print("    .config('spark.kryo.classesToRegister', 'mypackage.MyClass,mypackage.Other')")
print("    .getOrCreate()")
print()

# ── Conceptual timing comparison ─────────────────────────────────────────────
# We simulate the kind of RDD workload where Kryo helps most:
# shuffling custom Python objects (via rdd.map).

import pickle, io

# A representative "record" — the kind of object Kryo would handle
sample_records = [
    {"id": i, "name": f"user_{i}", "score": float(i % 100), "tags": [f"t{j}" for j in range(5)]}
    for i in range(10_000)
]

# Measure Python pickle serialisation time (analogous to Java serialiser)
t0 = time.time()
for rec in sample_records:
    buf = io.BytesIO()
    pickle.dump(rec, buf)
pickle_time = time.time() - t0

# Measure JSON serialisation time (lighter — analogous to what Kryo achieves
# for simple flat records)
import json
t0 = time.time()
for rec in sample_records:
    _ = json.dumps(rec).encode()
json_time = time.time() - t0

print(f"Python pickle (≈ Java serialiser)  : {pickle_time*1000:.1f} ms for 10k records")
print(f"JSON encode   (≈ Kryo, flat record): {json_time*1000:.1f} ms for 10k records")
print()
print("In practice, Kryo's advantage over Java serialiser is 2–5x for JVM objects.")
print("For Spark DataFrames (Tungsten), there is no benefit — Kryo only helps RDDs.")

In [ ]:
# Clean up and stop the session
spark.catalog.clearCache()
spark.stop()
print("SparkSession stopped. Notebook complete.")